# 05 — Demo end-to-end và so sánh hai mô hình

Phụ trách: **Thành viên 4** (luồng) + **Thành viên 5** (tổng hợp so sánh). Chạy `process_media` cho cùng một video bằng cả YOLO và Faster R-CNN rồi so sánh. **Không train model** ở notebook này: dùng checkpoint trong `models/yolo/best.pt` và `models/faster_rcnn/best.pth`.

## 0) Bootstrap (local / Kaggle / Colab)

- **Local**: notebook nằm trong repo nên dùng luôn repo hiện tại (không đụng vào git của bạn).
- **Kaggle / Colab**: clone repo về `/kaggle/working`, lấy đúng nhánh rồi cài thư viện. Trên Kaggle bật **Internet: On**.
- Repo private: xem `README.md` mục 12.8 (token lưu trong Kaggle Secrets).

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/ThanhBinh2006/VN-TSD-Voice-Alert.git"
REPO_DIR = "VN-TSD-Voice-Alert"
REPO_BRANCH = "main"              # Thành viên 1: data-yolo · Thành viên 2: faster-rcnn (khi nhánh đã có trên origin)

ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB = "google.colab" in sys.modules

def sh(cmd: str, check: bool = True) -> int:
    print(">>", cmd)
    return subprocess.run(cmd, shell=True, check=check).returncode

# 1. Tìm repo đang chứa notebook này (local)
local_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "ts_alert").is_dir() and (p / "configs" / "shared.yaml").exists()), None)

if local_root is not None:
    repo_path = local_root
    print("Dùng repo hiện tại:", repo_path)
else:
    # 2. Chưa có repo: clone rồi lấy đúng nhánh (chỉ làm trong môi trường cloud)
    assert REPO_URL != "TODO_URL", "Điền REPO_URL ở ô này trước khi chạy trên Kaggle/Colab."
    base = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
    repo_path = base / REPO_DIR
    if not (repo_path / ".git").exists():
        sh(f'git clone --branch {REPO_BRANCH} {REPO_URL} "{repo_path}"')
    git = f'git -C "{repo_path}"'
    sh(f"{git} fetch origin {REPO_BRANCH}")
    sh(f"{git} checkout -B {REPO_BRANCH} origin/{REPO_BRANCH}")
    sh(f"{git} log --oneline -1")

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
print("cwd:", Path.cwd())

# Cài thư viện: chỉ trên Kaggle/Colab. Local thì dùng môi trường đã cài bằng requirements.txt.
if ON_KAGGLE or ON_COLAB:
    sh('pip install -q -e ".[detect]"')

import torch
import ts_alert
print(f"Bootstrap OK. PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")

In [ ]:
RUNTIME = "configs/runtime/kaggle.yaml" if ON_KAGGLE else "configs/runtime/local.yaml"
print("Runtime:", RUNTIME)

## 1) Chọn video đầu vào

TODO: đặt một video thử (có biển báo ở nhiều vị trí trái/phải/trước mặt) vào `data/` rồi sửa đường dẫn.

In [ ]:
from ts_alert.config.resolve import resolve_config, resolve_path

VIDEO = resolve_path("data/sample.mp4")      # TODO: đổi đường dẫn
assert VIDEO.exists(), f"Chưa có video thử: {VIDEO}"

for model_name in ("yolo", "faster_rcnn"):
    weights = resolve_path(resolve_config(model_name, RUNTIME)["model"]["weights"])
    print(f"{model_name:12s} checkpoint:", weights, "| tồn tại:", weights.exists())

## 2) Chạy pipeline với từng mô hình

Tương đương dòng lệnh: `python -m tools.run_pipeline --input <video> --model yolo`.

In [ ]:
from ts_alert.pipeline import process_media

results = {}
for model_name in ("yolo", "faster_rcnn"):
    cfg = resolve_config(model_name, RUNTIME)
    results[model_name] = process_media(VIDEO, model_name, cfg=cfg)
    print(model_name, "->", results[model_name]["output_path"])

## 3) Bảng so sánh số liệu

In [ ]:
import pandas as pd

cols = ["unique_sign_count", "voice_play_count", "layer_time_ms", "tts_time_ms", "total_time_ms"]
pd.DataFrame({m: {c: r[c] for c in cols} for m, r in results.items()})

## 4) Xem video kết quả

In [ ]:
from IPython.display import Video, display

for model_name, r in results.items():
    print(model_name)
    display(Video(r["output_path"], embed=False, width=640))

## 5) Nghe lại các câu cảnh báo đã tạo

In [ ]:
from IPython.display import Audio

for model_name, r in results.items():
    print(model_name, f"({len(r['audio_paths'])} đoạn)")
    for path in r["audio_paths"]:
        print(" ", path)
        display(Audio(path))

## 6) Kiểm tra chống lặp

Với video có biển xuất hiện nhiều khung hình liên tiếp, `voice_play_count` phải nhỏ hơn rất nhiều so với số khung hình có phát hiện.

In [ ]:
for model_name, r in results.items():
    print(f"{model_name}: {r['unique_sign_count']} biển báo -> {r['voice_play_count']} lần đọc")

## 7) Nhận xét

TODO: điền — mô hình nào bỏ sót/nhận nhầm biển, độ trễ của layer và TTS so với tổng thời gian, chống lặp có đọc trùng hay bỏ sót biển nào không, giọng đọc tiếng Việt có tự nhiên không.